[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/miguepoloc/toma-decisiones-mcda/blob/main/01_ahp_cacao.ipynb)

# AHP, caso cacao

Zonificación de sensores IoT para Moniliasis en cacao, Sierra Nevada de Santa Marta, 4 zonas (Bonda, Guachaca, San Pedro, Palmor), 4 criterios (Temperatura, Humedad bajo el dosel, pH, Conductividad eléctrica). Contenido completo y verificado en la Sesión 2 (AHP) del curso.

Requiere `pip install pyDecision numpy`.

In [1]:
!pip install -q pyDecision


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import numpy as np
from pyDecision.algorithm import ahp_method

zonas = ["Bonda", "Guachaca", "San Pedro", "Palmor"]
criterios = ["Temperatura", "Humedad", "pH", "Conductividad"]

## Paso 1, Matriz de criterios

Juicios de Saaty (S2 §2.2): Humedad domina por ser el disparador de Moniliasis.

In [3]:
m_criterios = np.array([
    [1,   1/3, 1,   3],
    [3,   1,   3,   5],
    [1,   1/3, 1,   3],
    [1/3, 1/5, 1/3, 1],
])
w_criterios, cr_criterios = ahp_method(m_criterios, wd='m')
print("Pesos de criterios:", dict(zip(criterios, np.round(w_criterios, 4))))
print("CR:", round(cr_criterios, 4))

Pesos de criterios: {'Temperatura': np.float64(0.2009), 'Humedad': np.float64(0.5193), 'pH': np.float64(0.2009), 'Conductividad': np.float64(0.0789)}
CR: 0.0161


## Paso 2, Una matriz por criterio, comparando las 4 zonas

In [4]:
m_temp = np.array([
    [1,   2,   3,   5],
    [1/2, 1,   2,   4],
    [1/3, 1/2, 1,   3],
    [1/5, 1/4, 1/3, 1],
])
m_hum = np.array([
    [1,   2,   3,   5],
    [1/2, 1,   2,   3],
    [1/3, 1/2, 1,   2],
    [1/5, 1/3, 1/2, 1],
])
m_ph = np.array([
    [1,   2,   1/2, 3],
    [1/2, 1,   1/3, 2],
    [2,   3,   1,   4],
    [1/3, 1/2, 1/4, 1],
])
m_ce = np.array([
    [1,   4,   1/2, 3],
    [1/4, 1,   1/5, 1/2],
    [2,   5,   1,   3],
    [1/3, 2,   1/3, 1],
])

w_temp, cr_temp = ahp_method(m_temp, wd='m')
w_hum, cr_hum = ahp_method(m_hum, wd='m')
w_ph, cr_ph = ahp_method(m_ph, wd='m')
w_ce, cr_ce = ahp_method(m_ce, wd='m')

for nombre, w, cr in [("Temperatura", w_temp, cr_temp), ("Humedad", w_hum, cr_hum),
                       ("pH", w_ph, cr_ph), ("Conductividad", w_ce, cr_ce)]:
    print(f"{nombre:15s}", dict(zip(zonas, np.round(w, 4))), "CR=", round(cr, 4))

Temperatura     {'Bonda': np.float64(0.4709), 'Guachaca': np.float64(0.284), 'San Pedro': np.float64(0.1715), 'Palmor': np.float64(0.0736)} CR= 0.019
Humedad         {'Bonda': np.float64(0.4824), 'Guachaca': np.float64(0.2718), 'San Pedro': np.float64(0.1575), 'Palmor': np.float64(0.0883)} CR= 0.0054
pH              {'Bonda': np.float64(0.2771), 'Guachaca': np.float64(0.1611), 'San Pedro': np.float64(0.4658), 'Palmor': np.float64(0.096)} CR= 0.0115
Conductividad   {'Bonda': np.float64(0.3146), 'Guachaca': np.float64(0.0795), 'San Pedro': np.float64(0.4667), 'Palmor': np.float64(0.1392)} CR= 0.021


## Paso 3, Síntesis global

Prioridad global de cada zona = suma ponderada de sus prioridades locales por el peso de cada criterio.

In [5]:
prioridad_local = np.column_stack([w_temp, w_hum, w_ph, w_ce])
prioridad_global = prioridad_local @ w_criterios

print("Ranking final AHP:")
for z, p in sorted(zip(zonas, prioridad_global), key=lambda x: -x[1]):
    print(f"  {z:12s} {p:.4f}")

Ranking final AHP:
  Bonda        0.4256
  San Pedro    0.2466
  Guachaca     0.2368
  Palmor       0.0909


**Resultado esperado** (coincide con lo publicado en la Sesión 2 del curso): 1º Bonda (0.4266) · 2º San Pedro (0.2460) · 3º Guachaca (0.2370) · 4º Palmor (0.0904).

## Bonus: pesos calculados desde los datos (CRITIC, Entropía)

Sesión 1 diapositiva 5 anunció una 4ª vía para pesar criterios, calcularlo directamente de los datos en vez de por juicio experto (AHP, arriba). Sesión 4 diapositiva 19 desarrolla esto con números reales, sobre la misma matriz de decisión cruda de las 4 zonas y los mismos pesos 25/30/25/20 que usan `topsis_cacao.ipynb`/`vikor_cacao.ipynb`/`electre_cacao.ipynb`/`promethee_cacao.ipynb` para "AHP", **no** el `w_criterios` recién calculado arriba por comparación de a pares (20.1%/51.9%/20.1%/7.9%). Son dos resultados reales pero DISTINTOS para "el peso AHP" del mismo caso cacao — un desajuste ya existente entre este notebook y los otros 4 de esta carpeta, encontrado al ejecutar esta celda (no algo que esta adición haya causado), documentado aquí en vez de ocultado, y fuera de alcance resolver cuál de los dos es "el correcto" en esta misma pasada. Requiere `pip install pymcdm` además de pyDecision.

In [6]:
from pymcdm.helpers import normalize_matrix, correlation_matrix
from pymcdm.normalizations import minmax_normalization, sum_normalization
from pymcdm.correlations import pearson

dataset = np.array([
    [27, 78, 6.2, 0.9],   # Bonda
    [26, 85, 5.8, 1.5],   # Guachaca
    [24, 82, 6.5, 0.7],   # San Pedro
    [22, 88, 5.5, 1.2],   # Palmor
])
tipos = [1, -1, 1, -1]  # 1 = beneficio (Temp, pH), -1 = costo (Humedad, CE)
pesos_ahp_s3 = np.array([0.25, 0.30, 0.25, 0.20])  # ver nota arriba, no es w_criterios

**Gotcha real, verificado antes de confiar en el resultado:** `pymcdm.weights.critic_weights`/`entropy_weights` no reciben un parámetro de beneficio/costo, tratan las 4 columnas como beneficio por defecto (mismo tipo de error ya documentado en `promethee_cacao.ipynb` con `promethee_ii`). Por eso aquí se arma la normalización a mano, pasando `tipos` explícitamente, en vez de llamar `critic_weights(dataset)`/`entropy_weights(dataset)` directo — se comprobó que el resultado SÍ cambia sustancialmente si no se orienta antes (Temperatura pasa de 23.1% a 35.2%).

In [7]:
# CRITIC: variación (desviación estándar) x qué tan poco se parece a los demás
nmatrix_c = normalize_matrix(dataset, minmax_normalization, tipos)
std = nmatrix_c.std(axis=0, ddof=1)
coef = correlation_matrix(nmatrix_c, pearson, True)
w_critic = std * (1 - coef).sum(axis=0)
w_critic /= w_critic.sum()

# Entropía de Shannon: pesa más el criterio donde las alternativas más se distinguen
m, _ = dataset.shape
nmatrix_e = normalize_matrix(dataset, sum_normalization, tipos)
entropias = np.array([
    -np.sum(col * np.log(col)) if not np.any(col == 0) else 0.0
    for col in nmatrix_e.T
]) / np.log(m)
w_entropy = 1 - entropias
w_entropy /= w_entropy.sum()

print(f"{'Criterio':15s} {'AHP':>7s} {'CRITIC':>8s} {'Entropia':>9s}")
for nombre, wa, wc, we in zip(criterios, pesos_ahp_s3, w_critic, w_entropy):
    print(f"{nombre:15s} {wa:7.3f} {wc:8.3f} {we:9.3f}")

Criterio            AHP   CRITIC  Entropia
Temperatura       0.250    0.352     0.065
Humedad           0.300    0.155     0.021
pH                0.250    0.189     0.043
Conductividad     0.200    0.304     0.871


**Resultado esperado** (coincide con `sesion-04/README.md` diapositiva 19): Temperatura 0.250/0.352/0.065 · Humedad 0.300/0.155/0.021 · pH 0.250/0.189/0.043 · Conductividad 0.200/0.304/0.871 (AHP/CRITIC/Entropía). La Entropía concentra casi todo el peso en Conductividad por su alta variación relativa con solo 4 zonas, un resultado real, no un error de cálculo — debilidad conocida de la entropía de Shannon con pocas alternativas. Esta comparación es solo de PESOS, no se vuelve a correr el ranking completo con cada juego de pesos.